<a href="https://colab.research.google.com/github/shridhartapakire/movie-recommendation-system/blob/main/movie_recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Recommendation System

A content-based movie recommendation system using Natural Language Processing (NLP), text vectorization, and cosine similarity.

## Project Goals
- Explore and clean movie data.
- Build a content-based recommendation engine.
- Create a recommendation API using FastAPI.
- Develop a simple user interface.
- Track development using Git and GitHub.

In [36]:
# Check the Python environment
import sys

print("Python version:", sys.version)
print("Movie Recommendation System")
print("Day 1: Data Gathering and Cleaning")

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Movie Recommendation System
Day 1: Data Gathering and Cleaning


In [37]:
import pandas as pd

df = pd.read_csv("movies_metadata.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

df.head()

Dataset shape: (45466, 24)

Column names:
['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']


/tmp/ipykernel_3799/2078642297.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("movies_metadata.csv")


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [38]:
# Reload the dataset with consistent type inference
df = pd.read_csv("movies_metadata.csv", low_memory=False)

# Inspect missing values
print("Missing values in important columns:")
print(df[["title", "overview", "genres", "id"]].isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# Check data types
print("\nData types:")
print(df[["title", "overview", "genres", "id", "popularity"]].dtypes)

# Preview movie titles and descriptions
df[["title", "overview"]].head(10)

Missing values in important columns:
title         6
overview    954
genres        0
id            0
dtype: int64

Duplicate rows: 17

Data types:
title         object
overview      object
genres        object
id            object
popularity    object
dtype: object


,title,overview
0,Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...
2,Grumpier Old Men,A family wedding reignites the ancient feud be...
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...
5,Heat,"Obsessive master thief, Neil McCauley leads a ..."
6,Sabrina,An ugly duckling having undergone a remarkable...
7,Tom and Huck,"A mischievous young boy, Tom Sawyer, witnesses..."
8,Sudden Death,International action superstar Jean Claude Van...
9,GoldenEye,James Bond must unmask the mysterious head of ...


In [39]:
# Keep the original dataset unchanged
movies = df.copy()

# 1. Remove exact duplicate rows
movies = movies.drop_duplicates()

# 2. Remove rows with missing or blank titles
movies["title"] = movies["title"].astype("string").str.strip()

movies = movies[
    movies["title"].notna() &
    movies["title"].ne("")
].copy()

# 3. Handle missing overviews
movies["overview"] = movies["overview"].fillna("")

# 4. Reset row numbering after cleaning
movies = movies.reset_index(drop=True)

print("Original shape:", df.shape)
print("Cleaned shape:", movies.shape)
print("Remaining duplicate rows:", movies.duplicated().sum())
print("Missing titles:", movies["title"].isna().sum())
print("Missing overviews:", movies["overview"].isna().sum())

Original shape: (45466, 24)
Cleaned shape: (45443, 24)
Remaining duplicate rows: 0
Missing titles: 0
Missing overviews: 0


In [40]:
print(movies[["title", "overview", "genres", "id", "popularity"]].dtypes)

print("\nSample popularity values:")
print(movies["popularity"].head(10).tolist())

print("\nSample movie IDs:")
print(movies["id"].head(10).tolist())

title         string[python]
overview              object
genres                object
id                    object
popularity            object
dtype: object

Sample popularity values:
['21.946943', '17.015539', '11.7129', '3.859495', '8.387519', '17.924927', '6.677277', '2.561161', '5.23158', '14.686036']

Sample movie IDs:
['862', '8844', '15602', '31357', '11862', '949', '11860', '45325', '9091', '710']


In [41]:
# Convert popularity to numeric
movies["popularity"] = pd.to_numeric(
    movies["popularity"],
    errors="coerce"
)

# Keep movie IDs as strings
movies["id"] = movies["id"].astype("string")

print(movies[["title", "id", "popularity"]].dtypes)

print("\nMissing popularity values:", movies["popularity"].isna().sum())
print("\nDataset shape:", movies.shape)

title         string[python]
id            string[python]
popularity           float64
dtype: object

Missing popularity values: 0

Dataset shape: (45443, 24)


In [42]:
# Select the columns needed for content-based recommendations
movies["genres"] = movies["genres"].fillna("[]")
movies["overview"] = movies["overview"].fillna("")

print("Movies available:", len(movies))
print("\nSample movie text:")
print("Title:", movies.loc[0, "title"])
print("Overview:", movies.loc[0, "overview"])
print("Genres:", movies.loc[0, "genres"])

Movies available: 45443

Sample movie text:
Title: Toy Story
Overview: Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.
Genres: [{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]


In [43]:
import ast

def extract_genres(genre_text):
    try:
        genres = ast.literal_eval(genre_text) if isinstance(genre_text, str) else []
        return " ".join(
            genre["name"] for genre in genres
            if isinstance(genre, dict) and "name" in genre
        )
    except (ValueError, SyntaxError, TypeError):
        return ""

movies["genres_clean"] = movies["genres"].apply(extract_genres)

print("Title:", movies.loc[0, "title"])
print("Clean genres:", movies.loc[0, "genres_clean"])

Title: Toy Story
Clean genres: Animation Comedy Family


In [44]:
# Fill any remaining missing text values
movies["title"] = movies["title"].fillna("")
movies["overview"] = movies["overview"].fillna("")
movies["genres_clean"] = movies["genres_clean"].fillna("")

# Combine text features
movies["tags"] = (
    movies["title"] + " " +
    movies["overview"] + " " +
    movies["genres_clean"]
).str.strip()

# Inspect the result
print("Title:", movies.loc[0, "title"])
print("\nTags:")
print(movies.loc[0, "tags"])

Title: Toy Story

Tags:
Toy Story Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Animation Comedy Family


In [45]:
# Create a lowercase version of the movie text
movies["tags_lower"] = movies["tags"].str.lower()

print("Original tags:")
print(movies.loc[0, "tags"])

print("\nLowercase tags:")
print(movies.loc[0, "tags_lower"])

Original tags:
Toy Story Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Animation Comedy Family

Lowercase tags:
toy story led by woody, andy's toys live happily in his room until andy's birthday brings buzz lightyear onto the scene. afraid of losing his place in andy's heart, woody plots against buzz. but when circumstances separate buzz and woody from their owner, the duo eventually learns to put aside their differences. animation comedy family


In [46]:
from sklearn.feature_extraction.text import CountVectorizer

sample_text = movies.loc[0, "tags_lower"]

vectorizer = CountVectorizer()
vectorizer.fit([sample_text])

print("First 20 words in the vocabulary:")
print(vectorizer.get_feature_names_out()[:20])

First 20 words in the vocabulary:
['afraid' 'against' 'and' 'andy' 'animation' 'aside' 'birthday' 'brings'
 'but' 'buzz' 'by' 'circumstances' 'comedy' 'differences' 'duo'
 'eventually' 'family' 'from' 'happily' 'heart']


In [47]:
vectorizer = CountVectorizer(stop_words="english")
vectorizer.fit([sample_text])

print("Vocabulary after removing English stop words:")
print(vectorizer.get_feature_names_out()[:20])

Vocabulary after removing English stop words:
['afraid' 'andy' 'animation' 'aside' 'birthday' 'brings' 'buzz'
 'circumstances' 'comedy' 'differences' 'duo' 'eventually' 'family'
 'happily' 'heart' 'learns' 'led' 'lightyear' 'live' 'losing']


In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english", max_features=20)

sample_vector = tfidf.fit_transform([sample_text])

print("Vector shape:", sample_vector.shape)
print("\nVocabulary:")
print(tfidf.get_feature_names_out())

Vector shape: (1, 20)

Vocabulary:
['afraid' 'andy' 'animation' 'aside' 'birthday' 'brings' 'buzz'
 'circumstances' 'comedy' 'differences' 'duo' 'eventually' 'family'
 'happily' 'heart' 'learns' 'led' 'lightyear' 'live' 'woody']


In [49]:
# Check the text data before vectorization
print("Total movies:", len(movies))
print("Empty tags:", movies["tags"].str.strip().eq("").sum())
print("Missing tags:", movies["tags"].isna().sum())

# Preview a few movie titles and their tags
movies[["title", "tags"]].head(3)

Total movies: 45443
Empty tags: 0
Missing tags: 0


,title,tags
0,Toy Story,"Toy Story Led by Woody, Andy's toys live happi..."
1,Jumanji,Jumanji When siblings Judy and Peter discover ...
2,Grumpier Old Men,Grumpier Old Men A family wedding reignites th...


In [50]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert movie tags into numerical vectors
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(movies["tags_lower"])

print("Number of movies:", tfidf_matrix.shape[0])
print("Number of features:", tfidf_matrix.shape[1])
print("Matrix type:", type(tfidf_matrix).__name__)

Number of movies: 45443
Number of features: 5000
Matrix type: csr_matrix


In [51]:
from sklearn.metrics.pairwise import cosine_similarity

# Get Toy Story's vector
movie_index = 0
toy_story_vector = tfidf_matrix[movie_index]

# Compare Toy Story with all movies
similarity_scores = cosine_similarity(
    toy_story_vector,
    tfidf_matrix
).flatten()

print("Number of similarity scores:", len(similarity_scores))
print("Toy Story similarity:", similarity_scores[0])
print("Highest similarity score:", similarity_scores.max())

Number of similarity scores: 45443
Toy Story similarity: 1.0
Highest similarity score: 1.0


In [52]:
import numpy as np

# Sort movies by similarity, highest first
similar_movies = np.argsort(similarity_scores)[::-1]

# Exclude Toy Story itself and get the top 5
top_indices = [
    i for i in similar_movies
    if i != movie_index
][:5]

# Display recommendations
print("Movies recommended for Toy Story:\n")

for rank, idx in enumerate(top_indices, start=1):
    print(
        f"{rank}. {movies.iloc[idx]['title']} "
        f"(Similarity: {similarity_scores[idx]:.3f})"
    )

Movies recommended for Toy Story:

1. Toy Story 3 (Similarity: 0.521)
2. Andy Kaufman Plays Carnegie Hall (Similarity: 0.432)
3. The Champ (Similarity: 0.428)
4. The 40 Year Old Virgin (Similarity: 0.426)
5. Superstar: The Life and Times of Andy Warhol (Similarity: 0.421)


In [53]:
import numpy as np

def recommend_movies(movie_title, top_n=5):
    # Find the movie by title (case-insensitive)
    matches = movies.index[
        movies["title"].str.casefold() == movie_title.strip().casefold()
    ].tolist()

    if not matches:
        print(f"Movie '{movie_title}' was not found.")
        return

    movie_index = matches[0]

    # Compare this movie with all movies
    similarity_scores = cosine_similarity(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).flatten()

    # Sort by similarity, excluding the selected movie
    similar_indices = np.argsort(similarity_scores)[::-1]
    recommendations = [
        idx for idx in similar_indices if idx != movie_index
    ][:top_n]

    print(f"Movies recommended for {movies.iloc[movie_index]['title']}:\n")

    for rank, idx in enumerate(recommendations, start=1):
        print(
            f"{rank}. {movies.iloc[idx]['title']} "
            f"(Similarity: {similarity_scores[idx]:.3f})"
        )

In [54]:
recommend_movies("Jumanji", top_n=5)

Movies recommended for Jumanji:

1. Liar Game: Reborn (Similarity: 0.313)
2. Game Over (Similarity: 0.304)
3. Table No. 21 (Similarity: 0.282)
4. The Dark Angel (Similarity: 0.274)
5. Snowed Under (Similarity: 0.268)


In [55]:
# Build a second set of tags without the movie title
movies["content_tags"] = (
    movies["overview"].fillna("") + " " +
    movies["genres_clean"].fillna("")
).str.strip().str.lower()

# Build a TF-IDF matrix for the new features
content_tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

content_matrix = content_tfidf.fit_transform(movies["content_tags"])

print("Content matrix shape:", content_matrix.shape)

Content matrix shape: (45443, 5000)


In [56]:
jumanji_index = movies.index[
    movies["title"].str.casefold() == "jumanji"
][0]

content_scores = cosine_similarity(
    content_matrix[jumanji_index],
    content_matrix
).flatten()

content_indices = np.argsort(content_scores)[::-1]
content_recommendations = [
    i for i in content_indices if i != jumanji_index
][:5]

for rank, idx in enumerate(content_recommendations, start=1):
    print(
        f"{rank}. {movies.iloc[idx]['title']} "
        f"(Similarity: {content_scores[idx]:.3f})"
    )

1. Table No. 21 (Similarity: 0.305)
2. Liar Game: Reborn (Similarity: 0.303)
3. Word Wars (Similarity: 0.303)
4. The Dark Angel (Similarity: 0.281)
5. Snowed Under (Similarity: 0.276)


In [57]:
# Inspect Jumanji and its top recommendations
titles_to_check = [
    "Jumanji",
    "Table No. 21",
    "Liar Game: Reborn",
    "Word Wars"
]

for title in titles_to_check:
    row = movies[
        movies["title"].str.casefold() == title.casefold()
    ]

    if not row.empty:
        movie = row.iloc[0]
        print(f"\nTitle: {movie['title']}")
        print(f"Genres: {movie['genres_clean']}")
        print(f"Overview: {movie['overview'][:300]}")


Title: Jumanji
Genres: Adventure Fantasy Family
Overview: When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope for freedom is to finish the game, which proves risky as all thre

Title: Table No. 21
Genres: Adventure Thriller
Overview: A couple live a mediocre life and are thrilled to have won an exotic vacation to Fiji &amp; their excitement increases when they get a chance to play "Tell all truth" game for a mind boggling prize money. The game begins and the couple discovers that the game isn't really a game, but is a game of su

Title: Liar Game: Reborn
Genres: Drama
Overview: To exact revenge, the Liar Game office is revived. The target is only one person - Shinichi Akiyama (Shota Matsuda). Because of new heroine Shinomiya (Mikako Tabe), Shinichi, who kept refusing to take part in the game, finds himsel

In [58]:
# Give genres more weight than the overview
movies["weighted_tags"] = (
    movies["genres_clean"].fillna("") + " " +
    movies["genres_clean"].fillna("") + " " +
    movies["overview"].fillna("")
).str.lower()

weighted_tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

weighted_matrix = weighted_tfidf.fit_transform(
    movies["weighted_tags"]
)

print("Weighted matrix shape:", weighted_matrix.shape)

Weighted matrix shape: (45443, 10000)


In [59]:
jumanji_index = movies.index[
    movies["title"].str.casefold() == "jumanji"
][0]

weighted_scores = cosine_similarity(
    weighted_matrix[jumanji_index],
    weighted_matrix
).flatten()

weighted_indices = np.argsort(weighted_scores)[::-1]

weighted_recommendations = [
    i for i in weighted_indices if i != jumanji_index
][:5]

print("Recommendations for Jumanji:\n")

for rank, idx in enumerate(weighted_recommendations, start=1):
    print(
        f"{rank}. {movies.iloc[idx]['title']} "
        f"| Genres: {movies.iloc[idx]['genres_clean']} "
        f"| Similarity: {weighted_scores[idx]:.3f}"
    )


Recommendations for Jumanji:

1. Journey to the Center of the Earth | Genres: Adventure Fantasy Family | Similarity: 0.370
2. Harry Potter and the Chamber of Secrets | Genres: Adventure Fantasy Family | Similarity: 0.354
3. Aschenputtel | Genres: Fantasy Family | Similarity: 0.345
4. Harry Potter and the Half-Blood Prince | Genres: Adventure Fantasy Family | Similarity: 0.318
5. Clash of the Titans | Genres: Adventure Fantasy Family | Similarity: 0.316


In [60]:
def recommend_weighted(movie_title, top_n=5):
    matches = movies.index[
        movies["title"].str.casefold() == movie_title.casefold()
    ].tolist()

    if not matches:
        print(f"Movie not found: {movie_title}")
        return

    idx = matches[0]

    scores = cosine_similarity(
        weighted_matrix[idx],
        weighted_matrix
    ).flatten()

    ranked = np.argsort(scores)[::-1]
    ranked = [i for i in ranked if i != idx][:top_n]

    print(f"\nRecommendations for {movie_title}:")

    for rank, i in enumerate(ranked, 1):
        print(
            f"{rank}. {movies.iloc[i]['title']} "
            f"| Genres: {movies.iloc[i]['genres_clean']} "
            f"| Score: {scores[i]:.3f}"
        )


for title in ["Toy Story", "The Dark Knight", "Titanic"]:
    recommend_weighted(title)


Recommendations for Toy Story:
1. Toy Story 2 | Genres: Animation Comedy Family | Score: 0.528
2. Toy Story 3 | Genres: Animation Family Comedy | Score: 0.489
3. Superstar Goofy | Genres: Animation Comedy Family | Score: 0.365
4. Small Fry | Genres: Animation Family | Score: 0.332
5. Botsman i Popugay | Genres: Animation Comedy Family | Score: 0.330

Recommendations for The Dark Knight:
1. Ricochet | Genres: Action Crime Thriller | Score: 0.379
2. İtirazım Var | Genres: Drama Action Crime | Score: 0.366
3. The Dark Knight Rises | Genres: Action Crime Drama Thriller | Score: 0.362
4. Policewomen | Genres: Drama Action Crime | Score: 0.333
5. Underworld U.S.A. | Genres: Action Crime Thriller | Score: 0.330

Recommendations for Titanic:
1. Gone, But Not Forgotten | Genres: Drama Romance Thriller | Score: 0.231
2. Pola X | Genres: Drama Romance | Score: 0.225
3. Kill Your Darlings | Genres: Drama Romance Thriller | Score: 0.224
4. Fida | Genres: Thriller Drama Romance | Score: 0.221
5. Li

In [61]:
# Find titles that appear more than once
duplicate_titles = movies[
    movies["title"].str.casefold().duplicated(keep=False)
].sort_values("title")

print("Rows belonging to repeated titles:", len(duplicate_titles))
print("\nExamples:")
print(
    duplicate_titles[["title", "release_date", "genres_clean"]]
    .head(15)
    .to_string(index=False)
)

Rows belonging to repeated titles: 5574

Examples:
                       title release_date                             genres_clean
                  10 Minutes   2014-04-24                                         
                  10 Minutes   2002-01-01                                      War
                12 Angry Men   1957-03-25                                    Drama
                12 Angry Men   1997-08-17                              Crime Drama
                   12 Chairs   1971-06-21                         Adventure Comedy
                   12 Chairs   1976-12-31                         Adventure Comedy
20,000 Leagues Under the Sea   1954-12-23          Adventure Drama Science Fiction
20,000 Leagues Under the Sea   1997-03-23         Adventure Action Science Fiction
20,000 Leagues Under the Sea   1997-05-11 Adventure Fantasy Action Science Fiction
20,000 Leagues Under the Sea   1916-12-24   Adventure Drama Action Science Fiction
                        2:22   2008-

In [62]:
# Count distinct movie titles that occur more than once
title_counts = movies["title"].str.casefold().value_counts()

repeated_title_count = (title_counts > 1).sum()

print("Total unique titles:", movies["title"].str.casefold().nunique())
print("Titles appearing more than once:", repeated_title_count)
print("Rows belonging to repeated titles:", (title_counts[title_counts > 1]).sum())

Total unique titles: 42227
Titles appearing more than once: 2358
Rows belonging to repeated titles: 5574


In [63]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Prepare release years for display
movies["release_year"] = pd.to_datetime(
    movies["release_date"], errors="coerce"
).dt.year


def recommend_movies(movie_title, year=None, top_n=5):
    # Find movies matching the title
    matches = movies.index[
        movies["title"].str.casefold() == movie_title.strip().casefold()
    ].tolist()

    if not matches:
        print(f"Movie '{movie_title}' was not found.")
        return

    # If a year is provided, filter by that year
    if year is not None:
        matches = [
            idx for idx in matches
            if pd.notna(movies.loc[idx, "release_year"])
            and int(movies.loc[idx, "release_year"]) == year
        ]

        if not matches:
            print(f"No movie found with title '{movie_title}' from {year}.")
            return

    # If multiple versions remain, show available choices
    if len(matches) > 1:
        print(f"Multiple versions of '{movie_title}' found:")

        for idx in matches:
            release_year = movies.loc[idx, "release_year"]
            year_text = (
                str(int(release_year))
                if pd.notna(release_year)
                else "Unknown year"
            )
            print(f"- {movies.loc[idx, 'title']} ({year_text})")

        print("Please specify a year to select a version.")
        return

    movie_index = matches[0]

    # Calculate similarity against the selected movie
    scores = cosine_similarity(
        weighted_matrix[movie_index],
        weighted_matrix
    ).flatten()

    # Exclude the selected record and retrieve recommendations
    ranked_indices = np.argsort(scores)[::-1]
    ranked_indices = [
        idx for idx in ranked_indices if idx != movie_index
    ][:top_n]

    selected_title = movies.loc[movie_index, "title"]
    selected_year = movies.loc[movie_index, "release_year"]

    year_text = (
        str(int(selected_year))
        if pd.notna(selected_year)
        else "Unknown year"
    )

    print(f"Recommendations for {selected_title} ({year_text}):\n")

    for rank, idx in enumerate(ranked_indices, start=1):
        title = movies.loc[idx, "title"]
        release_year = movies.loc[idx, "release_year"]
        genres = movies.loc[idx, "genres_clean"]

        recommendation_year = (
            str(int(release_year))
            if pd.notna(release_year)
            else "Unknown year"
        )

        print(
            f"{rank}. {title} ({recommendation_year})"
            f" | Genres: {genres}"
            f" | Similarity: {scores[idx]:.3f}"
        )

In [64]:
recommend_movies("12 Angry Men", year=1957, top_n=5)

Recommendations for 12 Angry Men (1957):

1. Jury Duty (1995) | Genres: Comedy Drama | Similarity: 0.297
2. Mistrial (1996) | Genres: Thriller Drama Action | Similarity: 0.272
3. We Live Again (1934) | Genres: Drama | Similarity: 0.255
4. Murder! (1930) | Genres: Mystery Thriller | Similarity: 0.252
5. The Legend of Lizzie Borden (1975) | Genres: Drama | Similarity: 0.249


In [65]:
recommend_movies("12 Angry Men", year=1997, top_n=5)

Recommendations for 12 Angry Men (1997):

1. Le 7ème juré (2008) | Genres: Crime Drama | Similarity: 0.286
2. Las Poquianchis (1976) | Genres: Crime Drama | Similarity: 0.286
3. L'outremangeur (2002) | Genres: Crime Drama | Similarity: 0.286
4. The Testimony (1946) | Genres: Crime Drama | Similarity: 0.286
5. The Last Breath (1999) | Genres: Crime Drama | Similarity: 0.286


In [66]:
recommend_movies("Unknown Movie", top_n=5)

Movie 'Unknown Movie' was not found.


In [67]:
# Test 1: Unique title
recommend_movies("Toy Story", top_n=3)

# Test 2: Repeated title without a year
recommend_movies("12 Angry Men", top_n=3)

# Test 3: Existing title with a year that does not match
recommend_movies("12 Angry Men", year=1900, top_n=3)

Recommendations for Toy Story (1995):

1. Toy Story 2 (1999) | Genres: Animation Comedy Family | Similarity: 0.528
2. Toy Story 3 (2010) | Genres: Animation Family Comedy | Similarity: 0.489
3. Superstar Goofy (1991) | Genres: Animation Comedy Family | Similarity: 0.365
Multiple versions of '12 Angry Men' found:
- 12 Angry Men (1957)
- 12 Angry Men (1997)
Please specify a year to select a version.
No movie found with title '12 Angry Men' from 1900.


## Final Recommendation Engine

This section contains the selected recommendation approach after
testing multiple feature representations.

The final implementation uses genre-weighted movie text,
TF-IDF vectorization, and cosine similarity.

In [68]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Prepare the final text representation
final_movies = movies.copy()

final_movies["weighted_tags"] = (
    final_movies["genres_clean"].fillna("") + " " +
    final_movies["genres_clean"].fillna("") + " " +
    final_movies["overview"].fillna("")
).str.lower().str.strip()

# Build the final TF-IDF model
final_tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

final_matrix = final_tfidf.fit_transform(
    final_movies["weighted_tags"]
)

print("Final movies:", len(final_movies))
print("Final feature matrix:", final_matrix.shape)
print("Final model prepared successfully!")

Final movies: 45443
Final feature matrix: (45443, 10000)
Final model prepared successfully!


In [69]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def final_recommend_movies(movie_title, year=None, top_n=5):
    title = movie_title.strip().casefold()

    matches = final_movies.index[
        final_movies["title"].str.casefold() == title
    ].tolist()

    if not matches:
        print(f"Movie '{movie_title}' was not found.")
        return

    if year is not None:
        matches = [
            idx for idx in matches
            if pd.notna(final_movies.loc[idx, "release_year"])
            and int(final_movies.loc[idx, "release_year"]) == year
        ]

        if not matches:
            print(f"No movie found with title '{movie_title}' from {year}.")
            return

    if len(matches) > 1:
        print(f"Multiple versions of '{movie_title}' found:")

        for idx in matches:
            release_year = final_movies.loc[idx, "release_year"]
            year_text = (
                str(int(release_year))
                if pd.notna(release_year)
                else "Unknown year"
            )
            print(f"- {final_movies.loc[idx, 'title']} ({year_text})")

        print("Please specify a year to select a version.")
        return

    movie_index = matches[0]

    scores = cosine_similarity(
        final_matrix[movie_index],
        final_matrix
    ).ravel()

    scores[movie_index] = -1
    ranked_indices = np.argsort(scores)[::-1][:top_n]

    selected = final_movies.iloc[movie_index]
    selected_year = selected["release_year"]
    year_text = (
        str(int(selected_year))
        if pd.notna(selected_year)
        else "Unknown year"
    )

    print(f"Recommendations for {selected['title']} ({year_text}):\n")

    for rank, idx in enumerate(ranked_indices, start=1):
        movie = final_movies.iloc[idx]
        release_year = movie["release_year"]
        recommendation_year = (
            str(int(release_year))
            if pd.notna(release_year)
            else "Unknown year"
        )

        print(
            f"{rank}. {movie['title']} ({recommendation_year})"
            f" | Genres: {movie['genres_clean']}"
            f" | Similarity: {scores[idx]:.3f}"
        )

In [70]:
final_recommend_movies("Toy Story", year=1995, top_n=5)
final_recommend_movies("12 Angry Men", year=1997, top_n=5)
final_recommend_movies("Unknown Movie", top_n=3)

Recommendations for Toy Story (1995):

1. Toy Story 2 (1999) | Genres: Animation Comedy Family | Similarity: 0.528
2. Toy Story 3 (2010) | Genres: Animation Family Comedy | Similarity: 0.489
3. Superstar Goofy (1991) | Genres: Animation Comedy Family | Similarity: 0.365
4. Small Fry (2011) | Genres: Animation Family | Similarity: 0.332
5. Botsman i Popugay (1982) | Genres: Animation Comedy Family | Similarity: 0.330
Recommendations for 12 Angry Men (1997):

1. Bird of Prey (1995) | Genres: Crime Drama | Similarity: 0.286
2. The Last Breath (1999) | Genres: Crime Drama | Similarity: 0.286
3. The Testimony (1946) | Genres: Crime Drama | Similarity: 0.286
4. Las Poquianchis (1976) | Genres: Crime Drama | Similarity: 0.286
5. Le 7ème juré (2008) | Genres: Crime Drama | Similarity: 0.286
Movie 'Unknown Movie' was not found.


## Recommendation Evaluation

This section tests the recommendation engine with movies from different genres and checks whether recommendations are unique and exclude the selected movie.

In [72]:
# Test recommendations across different movies
test_movies = [
    ("Toy Story", 1995),
    ("Jumanji", 1995),
    ("The Dark Knight", 2008),
    ("Titanic", 1997),
]

for title, year in test_movies:
    print("=" * 60)
    final_recommend_movies(title, year=year, top_n=5)
    print()

Recommendations for Toy Story (1995):

1. Toy Story 2 (1999) | Genres: Animation Comedy Family | Similarity: 0.528
2. Toy Story 3 (2010) | Genres: Animation Family Comedy | Similarity: 0.489
3. Superstar Goofy (1991) | Genres: Animation Comedy Family | Similarity: 0.365
4. Small Fry (2011) | Genres: Animation Family | Similarity: 0.332
5. Botsman i Popugay (1982) | Genres: Animation Comedy Family | Similarity: 0.330

Recommendations for Jumanji (1995):

1. Journey to the Center of the Earth (1959) | Genres: Adventure Fantasy Family | Similarity: 0.370
2. Harry Potter and the Chamber of Secrets (2002) | Genres: Adventure Fantasy Family | Similarity: 0.354
3. Aschenputtel (2010) | Genres: Fantasy Family | Similarity: 0.345
4. Harry Potter and the Half-Blood Prince (2009) | Genres: Adventure Fantasy Family | Similarity: 0.318
5. Clash of the Titans (1981) | Genres: Adventure Fantasy Family | Similarity: 0.316

Recommendations for The Dark Knight (2008):

1. Ricochet (1991) | Genres: Actio

In [73]:
# Final validation checks

assert len(final_movies) == final_matrix.shape[0]
assert final_matrix.shape[1] == 10000
assert final_movies["title"].notna().all()
assert not final_movies["title"].str.strip().eq("").any()

print("All final validation checks passed!")
print("Movies:", len(final_movies))
print("TF-IDF features:", final_matrix.shape[1])
print("Recommendation engine is ready.")

All final validation checks passed!
Movies: 45443
TF-IDF features: 10000
Recommendation engine is ready.
